# PTB-XL: training-fold normalisation statistics (CPU)

Recomputes the channel-wise mean and standard deviation of the training folds (1–8), exactly as the main notebook does, and writes `reports/normalisation.json`. The released model needs these numbers to scale a new ECG the way it was trained. No training and no GPU: Kaggle settings **CPU**, **Internet on**.

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/ptbxl-ecg-superclass-benchmark.git", "/kaggle/working/ptbxl-bench"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/ptbxl-bench"], check=True)
import os
os.chdir("/kaggle/working/ptbxl-bench")
# The kernel started before the install, so add the package source to the path directly.
sys.path.insert(0, "/kaggle/working/ptbxl-bench/src")
import cardionet
print("cardionet", cardionet.__version__)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

## 1. PTB-XL 1.0.3
Uses an attached Kaggle copy if one is found under `/kaggle/input`; otherwise downloads the two metadata files and the 100 Hz records from PhysioNet (takes a while the first time).

In [ ]:
import glob, urllib.request, zipfile, pathlib, socket, time
BASE = "https://physionet.org/files/ptb-xl/1.0.3/"
socket.setdefaulttimeout(60)  # a stalled connection fails and is retried instead of hanging
found = glob.glob("/kaggle/input/**/ptbxl_database.csv", recursive=True)
if found:
    DATA_DIR = str(pathlib.Path(found[0]).parent)
else:
    # Official PhysioNet copy: the two metadata files plus only the 100 Hz records.
    DATA_DIR = "/kaggle/working/ptbxl"
    os.makedirs(DATA_DIR, exist_ok=True)
    for name in ["ptbxl_database.csv", "scp_statements.csv"]:
        urllib.request.urlretrieve(BASE + name, os.path.join(DATA_DIR, name))
    import pandas as pd
    from concurrent.futures import ThreadPoolExecutor

    def fetch(relative):
        target = os.path.join(DATA_DIR, relative)
        if not os.path.exists(target):
            os.makedirs(os.path.dirname(target), exist_ok=True)
            for attempt in range(10):
                try:
                    urllib.request.urlretrieve(BASE + relative, target + ".part")
                    os.replace(target + ".part", target)
                    break
                except Exception:
                    if attempt == 9:
                        raise
                    time.sleep(min(60, 2 ** attempt))  # PhysioNet returns 502/503 under load

    records = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"))["filename_lr"].tolist()
    files = [r + ext for r in records for ext in (".hea", ".dat")]
    with ThreadPoolExecutor(max_workers=8) as pool:
        for i, _ in enumerate(pool.map(fetch, files), 1):
            if i % 5000 == 0:
                print("downloaded", i, "of", len(files), flush=True)
    print("downloaded", len(files), "files")
import pandas as pd
db = pd.read_csv(os.path.join(DATA_DIR, "ptbxl_database.csv"))
print(DATA_DIR, "| records:", len(db), "| patients:", db.patient_id.nunique())

## 2. Prepare signals and write the statistics

In [ ]:
from cardionet.data import prepare_ptbxl, write_normalisation
import json, shutil
prepare_ptbxl(DATA_DIR, "data/processed", sampling_rate=100)
os.makedirs("/kaggle/working/reports", exist_ok=True)
stats = write_normalisation("data/processed", DATA_DIR, "/kaggle/working/reports/normalisation.json")
print(json.dumps(stats, indent=1))
shutil.rmtree("data/processed", ignore_errors=True)
shutil.rmtree("/kaggle/working/ptbxl", ignore_errors=True)